# NFQ

In [ ]:
!nvidia-smi

Mon Mar  9 00:54:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...    On  |   00000000:01:00.0  On |                  N/A |
| N/A   63C    P3             27W /   80W |     769MiB /   6144MiB |     36%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### CartPole-V1 Environment

In [3]:
env = gym.make("CartPole-v1")

In [4]:
env.action_space

Discrete(2)

In [5]:
env.observation_space

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)

In [6]:
env.reset()

(array([ 0.00878797, -0.04364273, -0.03330395,  0.00938301], dtype=float32),
 {})

In [7]:
env.action_space.n

np.int64(2)

## Imports

In [ ]:
import numpy as np

from tqdm import tqdm
import matplotlib.pyplot as plt

from itertools import count

import tempfile
import time
import glob
import gc
import random
import gymnasium as gym
from gymnasium import wrappers
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np

LEAVE_PRINT_EVERY_N_SECS = 60
ERASE_LINE = '\x1b[2K'
EPS = 1e-6
BEEP = lambda: os.system("printf '\a'")
RESULTS_DIR = os.path.join('..', 'results')
SEEDS = (12, 34, 56, 78, 90)


## Environment creation wrapper

In [2]:
env = gym.make('CartPole-v1')

In [3]:
env.unwrapped.reset(seed=33)

(array([-0.00563578,  0.00684912,  0.04081038, -0.02457505], dtype=float32),
 {})

In [4]:
env.reset(seed=33)

(array([-0.00563578,  0.00684912,  0.04081038, -0.02457505], dtype=float32),
 {})

In [5]:
def get_make_env_fn(**kwargs):
    def make_env_fn(env_name, seed=None, render=None, record=False, unwrapped=False, monitor_mode=None, inner_wrappers=None, outer_wrappers=None):
        mdir = tempfile.mkdtemp()
        env = None
        if render:
            try:
                env = gym.make(env_name, render=render)
            except:
                pass
        if env is None:
            env = gym.make(env_name)
        
        # if seed is not None: env.seed(seed)
        if seed is not None: env.unwrapped.reset(seed=seed)
        env = env.unwrapped if unwrapped else env
        if inner_wrappers:
            for wrapper in inner_wrappers:
                env = wrapper(env)
        env = wrappers.Monitor(env, mdir, force=True, mode=monitor_mode, video_callable=lambda e_idx: record) if monitor_mode else env
        if outer_wrappers:
            for wrapper in outer_wrappers:
                env = wrapper(env)
        return env
    return make_env_fn, kwargs

## Wrapper properly handling end of episode (failed or timeout termination)

In [6]:
class DiscountedCartPole(gym.Wrapper):
    def __init__(self, env):
        gym.Wrapper.__init__(self, env)
    def reset(self, **kwargs):
        return self.env.reset(**kwargs)
    def step(self, a):
        o, r, d, _ = self.env.step(a)
        (x, x_dot, theta, theta_dot) = o
        pole_fell = x < -self.env.unwrapped.x_threshold \
            or x > self.env.unwrapped.x_threshold \
            or theta < -self.env.unwrapped.theta_threshold_radians \
            or theta > self.env.unwrapped.theta_threshold_radians
        r = -1 if pole_fell else 0
        return o, r, d, _

## NFQ - Neural Fitted Q-learning Neural Network

In [71]:
class FCQ(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dims=(32,32), activation_fc=F.relu):
        super(FCQ, self).__init__()
        self.activation_fc = activation_fc
        self.input_layer = nn.Linear(input_dim, hidden_dims[0])
        self.hidden_layers = nn.ModuleList()
        for i in range(len(hidden_dims) - 1):
            hidden_layer = nn.Linear(hidden_dims[i], hidden_dims[i+1])
            self.hidden_layers.append(hidden_layer)
        self.output_layer = nn.Linear(hidden_dims[-1], output_dim)

        device = "cpu"
        if torch.cuda.is_available():
            device = "cuda:0"
        self.device = torch.device(device)
        self.to(self.device)

    
    def _format(self, state):
        x = state
        if not isinstance(x, torch.Tensor):
            x  = torch.Tensor(x).to(device=self.device)#,  dtype=torch.float32)            
            x = x.unsqueeze(0)
        return x

    def forward(self, state):
        x = self._format(state)
        x = self.activation_fc(self.input_layer(x))
        for hidden_layer in self.hidden_layers:
            x = self.activation_fc(hidden_layer(x))
        x = self.output_layer(x)
        return x

    def numpy_float_to_device(self, variable):
        variable = torch.from_numpy(variable).float().to(self.device)
        return variable

    def numpy_long_to_device(self, variable):
        variable = torch.from_numpy(variable).long().to(self.device)
        return variable

    def load(self, experiences):
        states, actions, rewards, new_states, is_terminals = experiences
        states = self.numpy_float_to_device(states)
        actions = self.numpy_long_to_device(actions)
        new_states = self.numpy_float_to_device(new_states)
        rewards = self.numpy_float_to_device(rewards)
        is_terminals = self.numpy_float_to_device(is_terminals)
        return states, actions, rewards, new_states, is_terminals
        

## Sampling strategies

In [72]:
class GreedyStrategy():
    def __init__(self):
        self.exploratory_action_taken = False

    def select_action(self, model, state):
        with torch.no_grad():
            q_values = model(state).cpu().detach().data.numpy().squeeze()
            return np.argmax(q_values)


In [73]:
class EGreedyStrategy():
    def __init__(self, epsilon=0.1):
        self.epsilon = epsilon
        self.exploratory_action_taken = None

    def select_action(self, model, state):
        self.exploratory_action_taken = False
        with torch.no_grad():
            q_values = model(state).cpu().detach().data.numpy().squeeze()

        if np.random.rand() > self.epsilon:
            action = np.argmax(q_values)
        else:
            action = np.random.randint(len(q_values))
        
        self.exploratory_action_taken = action != np.argmax(q_values)

        return action
        

## NFQ Class with all the RL related routines

In [ ]:
class NFQ():
    def __init__(self,
                value_model_fn,
                value_optimizer_fn,
                value_optimizer_lr,
                training_strategy_fn,
                evaluation_strategy_fn,
                batch_size,
                epochs):
        self.value_model_fn = value_model_fn
        self.value_optimizer_fn = value_optimizer_fn
        self.value_optimizer_lr = value_optimizer_lr
        self.training_strategy_fn = training_strategy_fn
        self.evaluation_strategy_fn = evaluation_strategy_fn
        self.batch_size = batch_size
        self.epochs = epochs

    def optimize_model(self, experiences):
        states, actions, rewards, next_states, is_terminal = experiences
        batch_size = len(is_terminal)

        max_a_q_sp = self.online_model(next_states).detach().max(1)[0].unsqueeze(1)
        target_q_s = rewards + self.gamma * max_a_q_sp * (1 - is_terminal)
        q_sa = self.online_model(states).gather(1, actions)

        td_errors = q_sa - target_q_s
        value_loss = td_errors.pow(2).mul(0.5).mean()
        self.value_optimizer.zero_grad()
        value_loss.backward()
        self.value_optimizer.step()

    def interaction_step(self, state, env):
        action = self.training_strategy.select_action(self.online_model, state)
        new_state, reward, is_terminal, _, info = env.step(action)
        is_truncated = 'TimeLimit.truncated' in info and info['TimeLimit.truncated']
        is_failure = is_terminal and not is_truncated
        experience = (state, action, reward, new_state, float(is_failure))

        self.experiences.append(experience)
        self.episode_reward[-1] += reward
        self.episode_timestep[-1] += 1
        self.episode_exploration[-1] += int(self.training_strategy.exploratory_action_taken)
        return new_state, is_terminal

    def train(self, make_env_fn, make_env_kargs, seed, gamma, max_minutes, max_episodes, goal_mean_100_reward):
        training_start, last_debug_time = time.time(), float('-inf')

        self.checkpoint_dir = tempfile.mkdtemp()
        self.make_env_fn = make_env_fn
        self.make_env_kargs = make_env_kargs
        self.seed = seed
        self.gamma = gamma

        env = self.make_env_fn(**self.make_env_kargs, seed=self.seed)
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)
        random.seed(self.seed)

        nS, nA = env.observation_space.shape[0], env.action_space.n
        
        self.episode_timestep = []
        self.episode_reward = []
        self.episode_seconds = []
        self.evaluation_scores = []
        self.episode_exploration = []

        self.online_model = self.value_model_fn(nS, nA)
        self.value_optimizer = self.value_optimizer_fn(self.online_model, self.value_optimizer_lr)

        self.training_strategy = self.training_strategy_fn()
        self.evaluation_strategy = self.evaluation_strategy_fn()

        self.experiences = []

        result = np.empty((max_episodes, 5))
        result[:] = np.nan
        training_time = 0
        for episode in range(1, max_episodes + 1):
            episode_start = time.time()

            state, is_terminal = env.reset()[0], False
            self.episode_reward.append(0.0)
            self.episode_timestep.append(0.0)
            self.episode_exploration.append(0.0)

            for step in count():
                # Experience buffer is enlarged with new experience from the line below inside the interaction_step function
                state, is_terminal = self.interaction_step(state, env)
                if len(self.experiences) >= self.batch_size:
                    experiences = np.array(self.experiences, dtype="object")
                    batches = [np.vstack(sars) for sars in experiences.T]
                    experiences = self.online_model.load(batches)
                    for _ in range(self.epochs):
                        self.optimize_model(experiences)
                    self.experiences.clear()

                if is_terminal:
                    gc.collect()
                    break
            
            episode_elapsed = time.time() - episode_start
            self.episode_seconds.append(episode_elapsed)
            training_time += episode_elapsed
            evaluation_score, _ = self.evaluate(self.online_model, env)
            self.save_checkpoint(episode-1, self.online_model)

            # Time steps are added in the interaction function
            total_step = int(np.sum(self.episode_timestep))
            self.evaluation_scores.append(evaluation_score)

            mean_10_reward = np.mean(self.episode_reward[-10:])
            std_10_reward = np.std(self.episode_reward[-10:])
            mean_100_reward = np.mean(self.episode_reward[-100:])
            std_100_reward = np.std(self.episode_reward[-100:])
            mean_100_eval_score = np.mean(self.evaluation_scores[-100:])
            std_100_eval_score = np.std(self.evaluation_scores[-100:])
            lsd_100_exp_rat = np.array(self.episode_exploration[-100:]) / np.array(self.episode_timestep[-100:])
            mean_100_exp_rat = np.mean(lsd_100_exp_rat)
            std_100_exp_rat = np.std(lsd_100_exp_rat)

            wallclock_elapsed = time.time() - training_start
            result[-1] = total_step, mean_100_reward, mean_100_eval_score, training_time, wallclock_elapsed

            reached_debug_time = time.time() - last_debug_time >= LEAVE_PRINT_EVERY_N_SECS
            reached_max_minutes = wallclock_elapsed >= max_minutes * 60
            reached_max_episodes = episode >= max_episodes
            reached_goal_mean_reward = mean_100_eval_score >= goal_mean_100_reward
            training_is_over = reached_max_minutes or reached_max_episodes or reached_goal_mean_reward

            elapsed_str = time.strftime("%H:%M:%S", time.gmtime(time.time() - training_start))

            debug_message = 'elapsed time  {}, episode {:04}, total step {:06}, '
            debug_message += 'mean reward: <10> {:05.1f}\u00B1{:05.1f}, '
            debug_message += '<100> {:05.1f}\u00B1{:05.1f}, '
            debug_message += 'exploration ratio: <100> {:05.1f}\u00B1{:05.1f}, '
            debug_message += 'eval score <100> {:05.1f}\u00B1{:05.1f}'

            debug_message = debug_message.format(elapsed_str, episode-1, total_step, mean_10_reward, std_10_reward, 
                                                mean_100_reward, std_100_reward, mean_100_exp_rat, std_100_exp_rat, mean_100_eval_score, std_100_eval_score)

            print(debug_message, end='\r', flush=True)

            if reached_debug_time or training_is_over:
                print(ERASE_LINE + debug_message, flush=True)
                last_debug_time = time.time()
            if training_is_over:
                if reached_max_minutes: print(u'--> reached_max_minutes \u2715')
                if reached_max_episodes: print(u'--> reached_max_episodes \u2715')
                if reached_goal_mean_reward: print(u'--> reached_goal_mean_reward \u2715')
                break
        
        final_eval_score, score_std = self.evaluate(self.online_model, env, n_episodes=100)

        wallclock_time = time.time() - training_start

        print("Training complete.")
        print('Final evaluation score {:.2f}\u00B1{:.2f} in {:.2f}s training time, '
            ' {:.2f}s wall-clock time.\n'.format(final_eval_score, score_std, training_time, wallclock_time))
        
        env.close() ; del env

        self.get_cleaned_checkpoints()
        return result, final_eval_score, training_time, wallclock_time

    def evaluate(self, eval_policy_model, eval_env, n_episodes=1):
        rs = []
        for _ in range(n_episodes):
            s, d = eval_env.reset()[0], False
            rs.append(0)
            for _ in count():
                a = self.evaluation_strategy.select_action(eval_policy_model, s)
                s, r, d, _ , _= eval_env.step(a)
                rs[-1] += r
                if d: break
        return np.mean(rs), np.std(rs)

    def get_cleaned_checkpoints(self, n_checkpoints=5):
        try:
            return self.checkpoint_paths
        except AttributeError:
            self.checkpoint_paths = {}

        
        paths = glob.glob(os.path.join(self.checkpoint_dir, '*.tar'))
        paths_dic = {int(path.split('.')[-2]):path for path in paths}
        last_ep = max(paths_dic.keys())
        checkpoint_idxs = np.linspace(1, last_ep+1, n_checkpoints, endpoint=True, dtype=np.int32) - 1

        for idx, path in paths_dic.items():
            if idx in checkpoint_idxs:
                self.checkpoint_paths[idx] = path
            else:
                os.unlink(path)
        
        return self.checkpoint_paths

# Skipping model inference demo code

    def save_checkpoint(self, episode_idx, model):
        torch.save(model.state_dict(), os.path.join(self.checkpoint_dir, 'model.{}.tar'.format(episode_idx)))

    


## Training the NFQ agent

In [92]:
nfq_results = []
best_agent, best_eval_score = None, float('-inf')
for seed in SEEDS:
    environment_settings = {
        'env_name': 'CartPole-v1',
        'gamma': 1.00,
        'max_minutes': 20,
        'max_episodes': 10000,
        'goal_mean_10_reward': 475
    }

    value_model_fn = lambda nS, nA: FCQ(nS, nA, hidden_dims=(512, 128))
    # value_optimizer_fn = lambda net, lr: optim.Adam(net.parameters(), lr=lr)
    value_optimizer_fn = lambda net, lr: optim.RMSprop(net.parameters(), lr=lr)
    value_optimizer_lr = 0.0005

    training_strategy_fn = lambda: EGreedyStrategy(epsilon=0.5)
    # evaluation_strategy_fn = lambda: EGreedyStrategy(epsilon=0.05)
    evaluation_strategy_fn = lambda: GreedyStrategy()

    batch_size = 1024
    epochs = 40

    env_name, gamma, max_minutes, max_episodes, goal_mean_100_reward = environment_settings.values()
    agent = NFQ(value_model_fn,
                value_optimizer_fn,
                value_optimizer_lr,
                training_strategy_fn,
                evaluation_strategy_fn,
                batch_size,
                epochs)
    # make_env_fn, make_env_kargs = get_make_env_fn(env_name=env_name, addon_wrapers=[DiscountedCartPole,])
    make_env_fn, make_env_kargs = get_make_env_fn(env_name=env_name)
    result, final_eval_score, training_time, wallclock_time = agent.train(make_env_fn,
                                                                    make_env_kargs,
                                                                    seed,
                                                                    gamma,
                                                                    max_minutes,
                                                                    max_episodes,
                                                                    goal_mean_100_reward)
    nfq_results.append(result)
    if final_eval_score > best_eval_score:
        best_eval_score = final_eval_score
        best_agent = agent
nfq_results = np.array(nfq_results)
_ = BEEP




elapsed time  00:00:00, episode 0000, total step 000014, mean reward: <10> 014.0±000.0, <100> 014.0±000.0, exploration ratio: <100> 000.4±000.0, eval score <100> 011.0±000.0
elapsed time  00:01:00, episode 0504, total step 008932, mean reward: <10> 018.8±004.8, <100> 021.4±007.0, exploration ratio: <100> 000.2±000.1, eval score <100> 018.4±003.4
elapsed time  00:02:00, episode 0930, total step 023702, mean reward: <10> 017.8±002.7, <100> 017.3±005.2, exploration ratio: <100> 000.2±000.1, eval score <100> 015.9±002.1
elapsed time  00:03:00, episode 1323, total step 043522, mean reward: <10> 149.7±057.6, <100> 081.3±064.1, exploration ratio: <100> 000.3±000.1, eval score <100> 150.7±103.5
elapsed time  00:04:00, episode 1611, total step 078555, mean reward: <10> 214.1±113.6, <100> 142.5±092.6, exploration ratio: <100> 000.3±000.1, eval score <100> 410.9±222.9
elapsed time  00:04:03, episode 1621, total step 080213, mean reward: <10> 165.8±115.5, <100> 142.9±096.8, exploration ratio: <100

KeyboardInterrupt: 